## ***Preprocessing***
Cleaning well log data, handling missing values, engineering features, and splitting by well for geologically honest train/test evaluation.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

DATA_PATH    = "C:/Users/user/Desktop/SK/Data Science/Personal Projects/Well-Log-Analysis/data/raw"
FIGURES_PATH = "C:/Users/user/Desktop/SK/Data Science/Personal Projects/Well-Log-Analysis/outputs/figures"
PROCESSED_PATH = "C:/Users/user/Desktop/SK/Data Science/Personal Projects/Well-Log-Analysis/data/processed"


os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Ready.")

Ready.


In [2]:
df = pd.read_csv(
    os.path.join(DATA_PATH, "CSV_train.csv"),
    sep=';'
)

# Add readable lithology names immediately
LITHOLOGY_KEYS = {
    30000: 'Sandstone',
    65030: 'Sandstone/Shale',
    65000: 'Shale',
    80000: 'Marl',
    74000: 'Dolomite',
    70000: 'Limestone',
    70032: 'Chalk',
    88000: 'Halite',
    86000: 'Anhydrite',
    99000: 'Tuff',
    90000: 'Coal',
    93000: 'Basement',
}

LITHOLOGY_COL = 'FORCE_2020_LITHOFACIES_LITHOLOGY'
df['LITHOLOGY_NAME'] = df[LITHOLOGY_COL].map(LITHOLOGY_KEYS)

print(f"Loaded : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Wells  : {df['WELL'].nunique()}")

Loaded : 1,170,511 rows × 30 columns
Wells  : 98


### ***Drop Excluded Columns***

In [3]:
# Columns to drop based on EDA findings
DROP_COLS = [
    'SGR',        # 94.1% missing + redundant with GR
    'RMIC',       # 85.0% missing
    'DTS',        # 85.1% missing
    'ROPA',       # 83.6% missing
    'DCAL',       # 74.5% missing — redundant with CALI anyway
    'MUDWEIGHT',  # 73.0% missing
    'RXO',        # 72.0% missing
    'ROP',        # 54.3% missing — drilling parameter, not petrophysical
    'BS',         # redundant with CALI (correlation = -0.90)
]

# Only drop columns that actually exist
DROP_COLS = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=DROP_COLS)

print(f"Dropped {len(DROP_COLS)} columns.")
print(f"Remaining columns : {df.shape[1]}")
print(f"Remaining cols    : {list(df.columns)}")

Dropped 9 columns.
Remaining columns : 21
Remaining cols    : ['WELL', 'DEPTH_MD', 'X_LOC', 'Y_LOC', 'Z_LOC', 'GROUP', 'FORMATION', 'CALI', 'RSHA', 'RMED', 'RDEP', 'RHOB', 'GR', 'NPHI', 'PEF', 'DTC', 'SP', 'DRHO', 'FORCE_2020_LITHOFACIES_LITHOLOGY', 'FORCE_2020_LITHOFACIES_CONFIDENCE', 'LITHOLOGY_NAME']


### ***Handle Sential Values***

In [ ]:
# In the oil industry, missing log readings are often stored as large negative numbers like -999.25 or -9999 rather than NaN.
# These are called sentinel values. If we leave them in, the model will treat -999.25 as a real measurement and learn nonsense.

# Standard industry sentinel values
SENTINELS = [-999.25, -9999, -9999.0, -999, 9999, 9999.0]

# Define the log columns we are actually using as features
META_COLS = [
    'WELL', 'DEPTH_MD', 'GROUP', 'FORMATION',
    'X_LOC', 'Y_LOC', 'Z_LOC',
    LITHOLOGY_COL, 'LITHOLOGY_NAME',
    'FORCE_2020_LITHOFACIES_CONFIDENCE'
]

LOG_COLS = [c for c in df.columns if c not in META_COLS]

print(f"Log feature columns ({len(LOG_COLS)}): {LOG_COLS}")
print()

# Replace sentinel values with NaN
rows_before = df.shape[0]
for col in LOG_COLS:
    for sentinel in SENTINELS:
        mask = np.isclose(df[col], sentinel, atol=0.01)
        df.loc[mask, col] = np.nan

print("Sentinel values replaced with NaN.")
print()

# Verify — check min values after cleaning
print("Min values after sentinel removal:")
for col in LOG_COLS:
    mn = df[col].min()
    print(f"  {col:<12} : {mn:.3f}")

Log feature columns (11): ['CALI', 'RSHA', 'RMED', 'RDEP', 'RHOB', 'GR', 'NPHI', 'PEF', 'DTC', 'SP', 'DRHO']

Sentinel values replaced with NaN.

Min values after sentinel removal:
  CALI         : 2.344
  RSHA         : 0.000
  RMED         : -0.008
  RDEP         : 0.032
  RHOB         : 0.721
  GR           : 0.109
  NPHI         : -0.036
  PEF          : 0.100
  DTC          : 7.415
  SP           : -608.787
  DRHO         : -7429.339


### ***Handle Physical Impossibilities***

In [ ]:
# Some log readings are physically impossible. These come from tool malfunctions or borehole effects.
# We replace them with NaN so they don't corrupt the model.

physical_limits = {
    'GR'   : (0,    300),    # API units — cannot be negative
    'RHOB' : (1.0,  3.5),    # g/cc — rock density range
    'NPHI' : (-0.15, 1.0),   # fraction — porosity bounds
    'DTC'  : (40,   240),    # us/ft — sonic travel time
    'RDEP' : (0.01, 100000), # ohm·m — resistivity
    'RMED' : (0.01, 100000),
    'RSHA' : (0.01, 100000),
    'CALI' : (4,    30),     # inches — borehole diameter
    'PEF'  : (0,    15),     # barns/electron
    'SP'   : (-300, 100),    # mV — spontaneous potential
    'DRHO' : (-0.5, 0.5),    # g/cc — density correction
}

total_clipped = 0
for col, (lo, hi) in physical_limits.items():
    if col not in df.columns:
        continue
    before = df[col].notna().sum()
    df.loc[df[col] < lo, col] = np.nan
    df.loc[df[col] > hi, col] = np.nan
    after   = df[col].notna().sum()
    clipped = before - after
    total_clipped += clipped
    if clipped > 0:
        print(f"  {col:<8} : {clipped:>6,} values outside [{lo}, {hi}] → NaN")

print(f"\nTotal impossible values removed : {total_clipped:,}")

  GR       :    960 values outside [0, 300] → NaN
  RHOB     :      9 values outside [1.0, 3.5] → NaN
  DTC      :    232 values outside [40, 240] → NaN
  RMED     :      7 values outside [0.01, 100000] → NaN
  RSHA     :     18 values outside [0.01, 100000] → NaN
  CALI     :    880 values outside [4, 30] → NaN
  PEF      : 30,711 values outside [0, 15] → NaN
  SP       : 163,967 values outside [-300, 100] → NaN
  DRHO     : 10,631 values outside [-0.5, 0.5] → NaN

Total impossible values removed : 207,415


### ***Merge Rare Cases***

In [ ]:
# Basement has only 103 samples — far too few to train a classifier.
# We merge it into Shale, which is geologically reasonable: Basement at the bottom of a well section has similar log character to tight, low-porosity shale.

print("Class counts before merging:\n")
print(df['LITHOLOGY_NAME'].value_counts())

# Merge Basement → Shale
df.loc[df['LITHOLOGY_NAME'] == 'Basement',
       LITHOLOGY_COL]     = 65000
df.loc[df['LITHOLOGY_NAME'] == 'Basement',
       'LITHOLOGY_NAME']  = 'Shale'

print("\nAfter merging Basement → Shale:")
print(df['LITHOLOGY_NAME'].value_counts())
print(f"\nFinal number of classes : {df['LITHOLOGY_NAME'].nunique()}")

Class counts before merging:

LITHOLOGY_NAME
Shale              720803
Sandstone          168937
Sandstone/Shale    150455
Limestone           56320
Marl                33329
Tuff                15245
Chalk               10513
Halite               8213
Coal                 3820
Dolomite             1688
Anhydrite            1085
Basement              103
Name: count, dtype: int64

After merging Basement → Shale:
LITHOLOGY_NAME
Shale              720906
Sandstone          168937
Sandstone/Shale    150455
Limestone           56320
Marl                33329
Tuff                15245
Chalk               10513
Halite               8213
Coal                 3820
Dolomite             1688
Anhydrite            1085
Name: count, dtype: int64

Final number of classes : 11


### ***Feature Engineering***

In [ ]:
# Feature engineering — create domain-informed derived features. These are standard petrophysical attributes that geoscientists compute routinely. Adding them gives the model variables that are directly interpretable in geological terms.

print("Engineering features...\n")

# 1. Neutron-Density separation
# When NPHI is high and RHOB is low simultaneously, the rock is porous.
# When they cross over (NPHI - RHOB crossover), it often indicates gas.
# This crossplot is the single most important gas indicator in petrophysics.
if 'NPHI' in df.columns and 'RHOB' in df.columns:
    df['NPHI_RHOB_SEP'] = df['NPHI'] - (df['RHOB'] - 1.0) / 1.65
    print("  NPHI_RHOB_SEP  : neutron-density separation (gas indicator)")

# 2. Resistivity ratio — shallow vs deep
# Deep resistivity reads the undisturbed formation.
# Shallow resistivity reads the zone invaded by drilling fluid.
# Their ratio indicates whether hydrocarbons are present.
if 'RDEP' in df.columns and 'RSHA' in df.columns:
    df['RES_RATIO'] = np.log10(df['RDEP'] / (df['RSHA'] + 1e-6))
    print("  RES_RATIO      : log10(RDEP/RSHA) — invasion indicator")

# 3. Density porosity
# Standard petrophysical formula using quartz grain density (2.65 g/cc)
# and fluid density (1.0 g/cc for freshwater).
# DPHI = (rho_matrix - RHOB) / (rho_matrix - rho_fluid)
if 'RHOB' in df.columns:
    RHO_MATRIX = 2.65   # quartz
    RHO_FLUID  = 1.0    # water
    df['DPHI'] = (RHO_MATRIX - df['RHOB']) / (RHO_MATRIX - RHO_FLUID)
    df['DPHI'] = df['DPHI'].clip(0, 0.5)
    print("  DPHI           : density porosity (standard formula)")

# 4. Vshale from GR — volume of shale
# Linear GR method: normalise GR between clean sand and pure shale endpoints.
# Vshale = 0 means clean sand, Vshale = 1 means pure shale.
if 'GR' in df.columns:
    GR_CLEAN = df['GR'].quantile(0.05)   # clean sand baseline
    GR_SHALE = df['GR'].quantile(0.95)   # pure shale baseline
    df['VSHALE'] = (df['GR'] - GR_CLEAN) / (GR_SHALE - GR_CLEAN)
    df['VSHALE'] = df['VSHALE'].clip(0, 1)
    print("  VSHALE         : volume of shale from GR (0=clean, 1=shale)")

# 5. Depth-normalised GR
# GR tends to increase slightly with depth due to compaction.
# Normalising removes this depth trend and isolates lithology signal.
if 'GR' in df.columns:
    df['GR_NORM'] = df.groupby('WELL')['GR'].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-6)
    )
    print("  GR_NORM        : depth-normalised GR per well")

# 6. Rolling window statistics along depth
# A single depth sample knows only its own value.
# The surrounding 5 samples (above and below) give context
# about whether we are in the middle of a thick layer or at a boundary.
if 'GR' in df.columns:
    df['GR_ROLLING_MEAN'] = df.groupby('WELL')['GR'].transform(
        lambda x: x.rolling(window=5, center=True, min_periods=1).mean()
    )
    df['GR_ROLLING_STD'] = df.groupby('WELL')['GR'].transform(
        lambda x: x.rolling(window=5, center=True, min_periods=1).std()
    ).fillna(0)
    print("  GR_ROLLING_MEAN: 5-sample rolling mean of GR")
    print("  GR_ROLLING_STD : 5-sample rolling std of GR")

print("\nFeature engineering complete.")
print(f"Total columns now : {df.shape[1]}")

Engineering features...

  NPHI_RHOB_SEP  : neutron-density separation (gas indicator)
  RES_RATIO      : log10(RDEP/RSHA) — invasion indicator
  DPHI           : density porosity (standard formula)
  VSHALE         : volume of shale from GR (0=clean, 1=shale)
  GR_NORM        : depth-normalised GR per well
  GR_ROLLING_MEAN: 5-sample rolling mean of GR
  GR_ROLLING_STD : 5-sample rolling std of GR

Feature engineering complete.
Total columns now : 28


### ***Create Label Encoding***

In [8]:
# Remap lithology names to integer class indices starting from 0.
# Same reason as Project A — models need 0-indexed integers.

unique_liths = sorted(df['LITHOLOGY_NAME'].unique())
LITH_TO_IDX  = {name: i for i, name in enumerate(unique_liths)}
IDX_TO_LITH  = {i: name for name, i in LITH_TO_IDX.items()}

df['LITH_IDX'] = df['LITHOLOGY_NAME'].map(LITH_TO_IDX)

print("Label encoding:\n")
for name, idx in LITH_TO_IDX.items():
    cnt = (df['LITH_IDX'] == idx).sum()
    print(f"  {idx} → {name:<20} ({cnt:,} samples)")

print(f"\nNumber of classes : {len(LITH_TO_IDX)}")

Label encoding:

  0 → Anhydrite            (1,085 samples)
  1 → Chalk                (10,513 samples)
  2 → Coal                 (3,820 samples)
  3 → Dolomite             (1,688 samples)
  4 → Halite               (8,213 samples)
  5 → Limestone            (56,320 samples)
  6 → Marl                 (33,329 samples)
  7 → Sandstone            (168,937 samples)
  8 → Sandstone/Shale      (150,455 samples)
  9 → Shale                (720,906 samples)
  10 → Tuff                 (15,245 samples)

Number of classes : 11


### ***Define Feature Sets***

In [9]:
# These are the columns the models will train on.
# We deliberately exclude metadata (location, depth, well name)
# because we want the model to learn from log physics,
# not from which well or depth it came from.
# Depth trends are partially captured by GR_NORM already.

FEATURE_COLS = (
    # Original log curves
    ['GR', 'RHOB', 'NPHI', 'DTC', 'RDEP', 'RMED',
     'RSHA', 'CALI', 'PEF', 'SP', 'DRHO']
    +
    # Engineered features
    [c for c in ['NPHI_RHOB_SEP', 'RES_RATIO', 'DPHI',
                 'VSHALE', 'GR_NORM',
                 'GR_ROLLING_MEAN', 'GR_ROLLING_STD']
     if c in df.columns]
)

# Only keep columns that actually exist
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

TARGET_COL = 'LITH_IDX'

print(f"Final feature set ({len(FEATURE_COLS)} features):\n")
for i, col in enumerate(FEATURE_COLS):
    origin = "engineered" if col not in LOG_COLS else "original log"
    print(f"  {i+1:>2}. {col:<20} ({origin})")

Final feature set (18 features):

   1. GR                   (original log)
   2. RHOB                 (original log)
   3. NPHI                 (original log)
   4. DTC                  (original log)
   5. RDEP                 (original log)
   6. RMED                 (original log)
   7. RSHA                 (original log)
   8. CALI                 (original log)
   9. PEF                  (original log)
  10. SP                   (original log)
  11. DRHO                 (original log)
  12. NPHI_RHOB_SEP        (engineered)
  13. RES_RATIO            (engineered)
  14. DPHI                 (engineered)
  15. VSHALE               (engineered)
  16. GR_NORM              (engineered)
  17. GR_ROLLING_MEAN      (engineered)
  18. GR_ROLLING_STD       (engineered)


### ***Split by Well***

In [10]:
# CRITICAL: we split by entire wells — not by random rows.
#
# Why? Consider a well with 20,000 depth samples.
# If we split randomly, training gets 18,000 samples from this well
# and test gets 2,000. The model has already seen the geological
# character of this well — it has learned its depth trends,
# its GR baseline, its porosity regime. The 2,000 test samples
# are trivially easy to predict because they come from a well
# the model has essentially memorised.
#
# Splitting by well forces the model to generalise to wells
# it has never seen at all — which is the real-world task.
# A petrophysicist trains on wells in one area and predicts
# lithology in a new well that hasn't been drilled yet.

all_wells = df['WELL'].unique()
np.random.shuffle(all_wells)

# Hold out 20% of wells for testing
n_test  = max(1, int(0.2 * len(all_wells)))
test_wells  = all_wells[:n_test]
train_wells = all_wells[n_test:]

df_train = df[df['WELL'].isin(train_wells)].copy()
df_test  = df[df['WELL'].isin(test_wells)].copy()

print(f"Total wells      : {len(all_wells)}")
print(f"Training wells   : {len(train_wells)}")
print(f"Test wells       : {len(test_wells)}")
print()
print(f"Training samples : {len(df_train):,}")
print(f"Test samples     : {len(df_test):,}")
print()
print(f"Test wells       : {list(test_wells)}")
print()

# Verify class distribution is reasonable in both splits
print("Class distribution — train vs test:\n")
print(f"  {'Class':<20}  {'Train%':>8}  {'Test%':>8}")
print("  " + "-"*40)

for name in sorted(LITH_TO_IDX.keys()):
    tr_pct = 100 * (df_train['LITHOLOGY_NAME'] == name).sum() / len(df_train)
    te_pct = 100 * (df_test['LITHOLOGY_NAME']  == name).sum() / len(df_test)
    print(f"  {name:<20}  {tr_pct:>7.1f}%  {te_pct:>7.1f}%")

Total wells      : 98
Training wells   : 79
Test wells       : 19

Training samples : 907,426
Test samples     : 263,085

Test wells       : ['34/10-33', '30/6-5', '35/9-6 S', '17/11-1', '35/11-12', '35/11-15 S', '34/11-1', '31/2-19 S', '16/2-11 A', '15/9-13', '25/6-3', '34/8-1', '31/3-2', '25/4-5', '31/2-8', '16/1-6 A', '25/2-13 T4', '16/2-6', '35/8-6 S']

Class distribution — train vs test:

  Class                   Train%     Test%
  ----------------------------------------
  Anhydrite                 0.1%      0.1%
  Chalk                     0.9%      1.0%
  Coal                      0.4%      0.2%
  Dolomite                  0.1%      0.2%
  Halite                    0.5%      1.5%
  Limestone                 4.9%      4.6%
  Marl                      2.5%      4.1%
  Sandstone                14.5%     14.3%
  Sandstone/Shale          13.5%     10.6%
  Shale                    61.2%     62.8%
  Tuff                      1.5%      0.6%


### ***Extract Feature and Target Matrices***

In [18]:
X_train = df_train[FEATURE_COLS].values.astype(np.float32)
y_train = df_train[TARGET_COL].values.astype(np.int32)

X_test  = df_test[FEATURE_COLS].values.astype(np.float32)
y_test  = df_test[TARGET_COL].values.astype(np.int32)



In [19]:
# ── Diagnose where the NaNs are coming from ──────────────────
print("NaN count per feature column:\n")
nan_df = pd.DataFrame({
    'Feature'  : FEATURE_COLS,
    'NaN_train': np.isnan(X_train).sum(axis=0),
    'NaN_test' : np.isnan(X_test).sum(axis=0),
})
nan_df['%_train'] = 100 * nan_df['NaN_train'] / len(X_train)
nan_df['%_test']  = 100 * nan_df['NaN_test']  / len(X_test)
nan_df = nan_df[nan_df['NaN_train'] > 0].sort_values('NaN_train', ascending=False)
print(nan_df.to_string(index=False))

NaN count per feature column:

        Feature  NaN_train  NaN_test   %_train    %_test
            PEF     400875    128655 44.177156 48.902446
      RES_RATIO     391405    151973 43.133545 57.765741
           RSHA     388098    151781 42.769107 57.692761
  NPHI_RHOB_SEP     363217     56410 40.027176 21.441739
           NPHI     350186     54916 38.591136 20.873862
             SP     314405    156228 34.648004 59.383089
           DRHO     146500     46785 16.144567 17.783226
           RHOB     133564     27714 14.718996 10.534238
           DPHI     133564     27714 14.718996 10.534238
           CALI      65691     23066  7.239268  8.767509
            DTC      56287     24808  6.202930  9.429652
           RMED      24112     14888  2.657186  5.659008
           RDEP      10814       201  1.191723  0.076401
        GR_NORM        232       728  0.025567  0.276717
             GR        232       728  0.025567  0.276717
         VSHALE        232       728  0.025567  0.276717


In [20]:
# ── Fix remaining NaNs with global median imputation ─────────
# Some wells have no reading at all for certain logs.
# Per-well median returns NaN for those wells.
# We now fill whatever remains with the global training median.

print("Fixing remaining NaN values...\n")

# Compute global medians from training set only
# (never use test set statistics — that would be data leakage)
train_medians = {}
for i, col in enumerate(FEATURE_COLS):
    col_vals = X_train[:, i]
    median   = np.nanmedian(col_vals)
    train_medians[col] = median

# Apply to training set
for i, col in enumerate(FEATURE_COLS):
    nan_mask = np.isnan(X_train[:, i])
    if nan_mask.sum() > 0:
        X_train[nan_mask, i] = train_medians[col]
        print(f"  X_train {col:<20} : filled {nan_mask.sum():,} NaN with {train_medians[col]:.4f}")

# Apply same medians to test set
# (use training medians — not test medians — to avoid leakage)
print()
for i, col in enumerate(FEATURE_COLS):
    nan_mask = np.isnan(X_test[:, i])
    if nan_mask.sum() > 0:
        X_test[nan_mask, i] = train_medians[col]
        print(f"  X_test  {col:<20} : filled {nan_mask.sum():,} NaN with {train_medians[col]:.4f}")

# Verify
print(f"\nNaN in X_train after fix : {np.isnan(X_train).sum()}")
print(f"NaN in X_test  after fix : {np.isnan(X_test).sum()}")

Fixing remaining NaN values...

  X_train GR                   : filled 232 NaN with 67.3588
  X_train RHOB                 : filled 133,564 NaN with 2.3203
  X_train NPHI                 : filled 350,186 NaN with 0.3246
  X_train DTC                  : filled 56,287 NaN with 109.4266
  X_train RDEP                 : filled 10,814 NaN with 1.4068
  X_train RMED                 : filled 24,112 NaN with 1.4256
  X_train RSHA                 : filled 388,098 NaN with 1.3562
  X_train CALI                 : filled 65,691 NaN with 12.6024
  X_train PEF                  : filled 400,875 NaN with 4.2861
  X_train SP                   : filled 314,405 NaN with 48.5462
  X_train DRHO                 : filled 146,500 NaN with 0.0011
  X_train NPHI_RHOB_SEP        : filled 363,217 NaN with -0.5313
  X_train RES_RATIO            : filled 391,405 NaN with -0.0262
  X_train DPHI                 : filled 133,564 NaN with 0.1998
  X_train VSHALE               : filled 232 NaN with 0.4251
  X_train GR_

In [21]:
print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}  unique classes: {np.unique(y_train)}")
print(f"X_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape}   unique classes: {np.unique(y_test)}")
print()

# Confirm no NaN values remain
print(f"NaN in X_train : {np.isnan(X_train).sum()}")
print(f"NaN in X_test  : {np.isnan(X_test).sum()}")

X_train : (907426, 18)
y_train : (907426,)  unique classes: [ 0  1  2  3  4  5  6  7  8  9 10]
X_test  : (263085, 18)
y_test  : (263085,)   unique classes: [ 0  1  2  3  4  5  6  7  8  9 10]

NaN in X_train : 0
NaN in X_test  : 0


Why use training medians for the test set? This is a data leakage rule. If you compute the median of the test set and use it to fill test set NaNs, you are using information from the test set during preprocessing — which artificially inflates test performance. The test set must be treated as if it doesn't exist until the final evaluation. So we compute medians only from training data and apply those same values to both sets.

In [22]:
print("Saving processed data...\n")

np.save(os.path.join(PROCESSED_PATH, "X_train.npy"),       X_train)
np.save(os.path.join(PROCESSED_PATH, "y_train.npy"),       y_train)
np.save(os.path.join(PROCESSED_PATH, "X_test.npy"),        X_test)
np.save(os.path.join(PROCESSED_PATH, "y_test.npy"),        y_test)

# Save mappings for later use in evaluation
np.save(os.path.join(PROCESSED_PATH, "lith_to_idx.npy"),   LITH_TO_IDX)
np.save(os.path.join(PROCESSED_PATH, "idx_to_lith.npy"),   IDX_TO_LITH)
np.save(os.path.join(PROCESSED_PATH, "feature_cols.npy"),  FEATURE_COLS)

# Save the full processed dataframes for well log plots in Phase 5
df_train.to_csv(os.path.join(PROCESSED_PATH, "df_train.csv"), index=False)
df_test.to_csv(os.path.join(PROCESSED_PATH, "df_test.csv"), index=False)

print("Saved files:")
for f in sorted(os.listdir(PROCESSED_PATH)):
    size = os.path.getsize(os.path.join(PROCESSED_PATH, f)) / 1e6
    print(f"  {f:<35} {size:>7.1f} MB")

Saving processed data...

Saved files:
  X_test.npy                             18.9 MB
  X_train.npy                            65.3 MB
  df_test.csv                            85.1 MB
  df_train.csv                          296.0 MB
  feature_cols.npy                        0.0 MB
  idx_to_lith.npy                         0.0 MB
  lith_to_idx.npy                         0.0 MB
  y_test.npy                              1.1 MB
  y_train.npy                             3.6 MB


In [16]:
print("=" * 55)
print("  PREPROCESSING COMPLETE")
print("=" * 55)
print(f"  Original rows          : 1,170,511")
print(f"  Final training samples : {len(X_train):,}")
print(f"  Final test samples     : {len(X_test):,}")
print(f"  Features               : {len(FEATURE_COLS)}")
print(f"  Classes                : {len(LITH_TO_IDX)}")
print(f"  Split strategy         : by well (not by row)")
print(f"  Imputation             : per-well median")
print(f"  Sentinel handling      : replaced with NaN then imputed")
print(f"  Class merging          : Basement → Shale")
print(f"  Engineered features    : NPHI_RHOB_SEP, RES_RATIO,")
print(f"                           DPHI, VSHALE, GR_NORM,")
print(f"                           GR_ROLLING_MEAN, GR_ROLLING_STD")
print("=" * 55)

  PREPROCESSING COMPLETE
  Original rows          : 1,170,511
  Final training samples : 907,426
  Final test samples     : 263,085
  Features               : 18
  Classes                : 11
  Split strategy         : by well (not by row)
  Imputation             : per-well median
  Sentinel handling      : replaced with NaN then imputed
  Class merging          : Basement → Shale
  Engineered features    : NPHI_RHOB_SEP, RES_RATIO,
                           DPHI, VSHALE, GR_NORM,
                           GR_ROLLING_MEAN, GR_ROLLING_STD
